# 03 · Segmentación de clientes con K-Means

**Objetivo:** agrupar a los clientes según su comportamiento de viaje y gasto para identificar segmentos.

**Datos:** `Clientes_Demorados_SinDuplicados.csv` (22.391 clientes). Se descartan `id_cliente`, `sexo`,
`provincia` y `canal_compra`; `programaMillas` se codifica 0/1; `ocupacion` y `clase_preferida` van con one-hot;
y todo se estandariza (K-Means es sensible a la escala).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from vuelos_ds import clustering, config, data, plots

plots.set_style()
from sklearn.metrics import silhouette_score

## 1. Datos y preparación

In [ ]:
clientes = data.load_clientes_demorados()
display(clientes.describe().round(2))

df_km, X = clustering.prepare_kmeans_features(clientes)
print("Dimensiones tras codificar y estandarizar:", X.shape)

## 2. Elección de K

In [ ]:
ks = range(1, 10)
plots.plot_elbow(ks, clustering.elbow(X, ks))
plt.show()

In [ ]:
sil = clustering.silhouettes(X, range(2, 6))
display(pd.Series(sil, name="silueta").round(4).to_frame())
plots.plot_silhouette(sil)
plt.show()

In [ ]:
for k in [2, 3, 4, 5]:
    print(f"\n{'=' * 40}\n K = {k}\n{'=' * 40}")
    tmp = df_km.assign(Cluster=clustering.fit_kmeans(X, k).labels_)
    display(clustering.perfil_clusters(tmp, ["cantidad_millas", "gasto_acumulado", "ingreso_mensual", "cant_vuelos"]))
    plots.plot_cluster_scatter(tmp, "cantidad_millas", "gasto_acumulado", ax=None, title=f"K = {k}: millas vs gasto")
    plt.show()

**Decisión:** se elige **K = 3**. La silueta es más alta con K = 2 (0,28), pero en 2 grupos se pierde el
segmento intermedio; K = 3 da una segmentación más útil para el negocio (ver sección 3), aunque la estructura de
clusters es débil.

## 3. Modelo final (K = 3)

In [ ]:
K = 3
modelo = clustering.fit_kmeans(X, K)
df_km["Cluster"] = modelo.labels_

score = silhouette_score(X, modelo.labels_)
print(f"Coeficiente de silueta (K={K}): {score:.4f}")
print("Interpretación:", clustering.interpretar_silueta(score))
display(df_km["Cluster"].value_counts().sort_index().to_frame("clientes"))

In [ ]:
poder = clustering.poder_discriminante(df_km)
display(poder.round(1).to_frame("Poder de separación %"))
plots.plot_bar_series(poder, "Poder discriminante de cada variable", "Variabilidad entre clusters (%)")
plt.show()

In [ ]:
display(clustering.perfil_clusters(df_km))

In [ ]:
for x, y in [("cantidad_millas", "gasto_acumulado"), ("cantidad_millas", "ingreso_mensual"),
             ("cant_vuelos", "gasto_acumulado"), ("ingreso_mensual", "anticipacion_compra_promedio"),
             ("edad", "ingreso_mensual")]:
    plots.plot_cluster_scatter(df_km, x, y)
    plt.show()

## 4. Conclusiones

*(Valores de la corrida original con los datos reales.)*

| Segmento | Vuelos | Gasto acum. | Millas | Ingreso mensual | En prog. de millas |
|---|---|---|---|---|---|
| **Premium / alta frecuencia** (cluster 1) | ~54 | ~$32.300 | ~48.400 | ~$7.600 | 61 % |
| **Intermedio** (cluster 2) | ~14 | ~$8.100 | ~4.800 | ~$4.300 | 20 % |
| **Bajo uso** (cluster 0) | ~13 | ~$2.800 | ~1.900 | ~$2.100 | 20 % |

- Las variables que más separan los grupos son **millas, gasto acumulado, gasto extra y cantidad de vuelos**;
  la **edad casi no discrimina** (3 %).
- **Limitación:** silueta de 0,17 → estructura débil, los grupos se solapan. Conviene tratar los segmentos como
  una descripción orientativa y no como categorías nítidas.
- Posibles mejoras: probar otros algoritmos (jerárquico, GMM), reducir dimensionalidad antes de clusterizar
  (ver `04_pca_clientes.ipynb`) y validar la estabilidad con distintas semillas.